In [36]:

from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [37]:
import pandas as pd

INPUT_PATH = "/content/drive/MyDrive/processed_dataset.xlsx"
df = pd.read_excel(INPUT_PATH)
print(df.shape)
df.head()




(10019, 36)


,order_id,supplier_id,supplier_rating,supplier_lead_time,order_date,promised_delivery_date,actual_delivery_date,shipping_distance_km,order_quantity,unit_price,...,region_South,region_West,holiday_period_Yes,carrier_name_DHL,carrier_name_Delhivery,carrier_name_EcomExpress,carrier_name_FedEx,delayed_reason_code_Operational,delayed_reason_code_Traffic,delayed_reason_code_Weather
0,1.0,5322.0,3.4,10,2024-05-15,2024-05-25,2024-05-29,51,48,2153.91,...,False,False,False,False,False,True,False,True,False,False
1,2.0,3932.0,4.3,10,2024-11-12,2024-11-22,2024-11-27,373,91,405.36,...,False,False,True,True,False,False,False,False,False,False
2,3.0,8966.0,3.2,5,2024-08-28,2024-09-02,2024-09-02,1304,25,3241.41,...,True,False,False,False,False,False,False,False,False,False
3,4.0,9832.0,3.9,7,2024-08-12,2024-08-19,2024-08-19,839,71,365.79,...,False,False,False,False,False,False,True,True,False,False
4,5.0,2126.0,3.2,8,2024-07-07,2024-07-15,2024-07-18,258,9,3052.84,...,False,False,False,False,False,False,False,False,False,False


In [38]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.utils.class_weight import compute_class_weight

from xgboost import XGBClassifier
import joblib


In [39]:
df["promised_delivery_date"] = pd.to_datetime(df["promised_delivery_date"])
df["actual_delivery_date"] = pd.to_datetime(df["actual_delivery_date"])

# Target: 1 = Delayed, 0 = On-Time
df["delayed"] = (df["actual_delivery_date"] > df["promised_delivery_date"]).astype(int)

df["delayed"].value_counts()


,count
delayed,
1,7159
0,2860


In [40]:
leakage_columns = [
    "order_id",
    "supplier_id",
    "order_date",
    "promised_delivery_date",
    "actual_delivery_date",
    "previous_on_time_rate",   # ← MAJOR LEAKAGE
    "delayed_reason_code_Operational",
    "delayed_reason_code_Traffic",
    "delayed_reason_code_Weather"
]

X = df.drop(columns=leakage_columns + ["delayed"], errors="ignore")
y = df["delayed"]

print("Features used:", X.shape[1])


Features used: 27


In [41]:
X = X.drop(columns=["previous_on_time_rate"], errors="ignore")


In [42]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [43]:
classes = np.array([0, 1])

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y
)

class_weight_dict = {0: class_weights[0], 1: class_weights[1]}

scale_pos_weight = class_weight_dict[0] / class_weight_dict[1]

scale_pos_weight


np.float64(2.5031468531468533)

In [44]:
model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42
)

model.fit(X_train, y_train)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=5, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None,
              num_parallel_tree=None, ...)

In [45]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


ROC-AUC: 0.9955109485486581

Confusion Matrix:
 [[ 559   13]
 [   4 1428]]

Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.98      0.99       572
           1       0.99      1.00      0.99      1432

    accuracy                           0.99      2004
   macro avg       0.99      0.99      0.99      2004
weighted avg       0.99      0.99      0.99      2004



In [71]:
joblib.dump(model, "shipment_delay_model.pkl")
joblib.dump(X.columns.tolist(), "model_features.pkl")


['model_features.pkl']